# 乱数シードを固定して実験を再現できるようにする

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mamorlis/nlpcookbook/blob/main/notebooks/01_colab_tips/set_seed.ipynb)

NLP クックブック 01-04 ／ Google Colab の小技

実行するたびに結果が変わってしまう。同じ結果を再現できるように、乱数を固定したい。

In [1]:
import os
import random
import numpy as np
import torch

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)

for trial in range(2):
    set_seed(0)
    print(random.random(), np.random.rand(), torch.rand(1).item())

0.8444218515250481 0.5488135039273248 0.49625658988952637
0.8444218515250481 0.5488135039273248 0.49625658988952637


In [2]:
# Transformers を使っているなら、これ 1 つで Python・NumPy・PyTorch をまとめて固定できる
from transformers import set_seed as hf_set_seed

hf_set_seed(0)
layer = torch.nn.Linear(4, 2)
print(layer.weight[0].tolist())
hf_set_seed(0)
print(torch.nn.Linear(4, 2).weight[0].tolist())

[-0.003743410110473633, 0.26822179555892944, -0.4115225672721863, -0.3679695129394531]
[-0.003743410110473633, 0.26822179555892944, -0.4115225672721863, -0.3679695129394531]


In [3]:
# GPU での完全な再現性が必要な場合 (遅くなることがある)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
# torch.use_deterministic_algorithms(True)   # 非決定的な演算があるとエラーにして知らせる

# グローバルな乱数に頼らず、専用の乱数生成器を使うのが安全
rng = np.random.default_rng(0)
g = torch.Generator().manual_seed(0)
print(rng.integers(0, 100, 3), torch.randperm(5, generator=g).tolist())

[85 63 51] [4, 0, 1, 3, 2]


- シードを固定しても、GPU の機種、ライブラリのバージョン、並列処理の順序が違えば結果は変わりうる。完全な再現より、「複数シードで傾向が変わらないこと」を確認するほうが大事。
- モデルの初期化の前、データのシャッフルの前など、乱数を使う処理より前に呼ぶ。セルを実行し直すと乱数の状態が進むので、必要なら再度呼ぶ。
- `DataLoader` で `num_workers > 0` のときは、`worker_init_fn` と `generator` も指定する。